Isolation forest with different feature combinations

In [ ]:
from scipy.stats import skew
import matplotlib.pyplot as plt
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import classification_report, f1_score

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
root_dir = r'smaller_dataset.csv'
df = pd.read_csv(root_dir)

Preprocessing

In [ ]:
attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Uploading_Attack"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

In [ ]:
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"] 

for col in categorical_columns:
    df[col] = df[col].astype('category')

In [ ]:
for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

In [ ]:
def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign&Bruteforce_benign':  
        return 'Normal'
    else:
        return 'Attack'

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

print(df[['Attack_Type', 'Anomaly_Label']].head())

In [ ]:
df.columns

In [ ]:
null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]  


inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]

df = df.dropna()

df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]

df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
X = df.select_dtypes(include=['number', 'bool'])
y = df['Anomaly_Label'] 
y = (y == "Attack").astype(int)  # Attack = 1, Normal = 0

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Train/Test splitting the dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:

RANDOM_STATE = 42
# Mutual Information
mi_scores = mutual_info_classif(X_scaled, y, random_state=RANDOM_STATE)
mi_ranking = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# Random Forest Importance
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_scaled, y)
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

In [ ]:
print("\n--- Mutual Information Feature Ranking ---")
for i, (feature, score) in enumerate(mi_ranking.items(), 1):
    print(f"{i:2d}. {feature:<30} Score: {score:.4f}")

print("\n--- Random Forest Feature Importance ---")
for i, (feature, score) in enumerate(rf_importance.items(), 1):
    print(f"{i:2d}. {feature:<30} Importance: {score:.4f}")

In [ ]:
from itertools import combinations
import csv

In [ ]:
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

finding the intersection of top k features in feature rankings, and then trying out Iforest with each

In [ ]:
TOP_K_LIST = [5, 10, 20]
OUTPUT_FILE = "iforest_results.csv"

with open(OUTPUT_FILE, mode='w', newline='') as file:
    fieldnames = ["Top_K", "Feature_Combination_Size", "Features", "ROC_AUC", "Precision", "Recall", "F1_Score", "Accuracy"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()


    for k in TOP_K_LIST:
        mi_top_k = set(mi_ranking.head(k).index)
        rf_top_k = set(rf_importance.head(k).index)
        combined_features = list(mi_top_k & rf_top_k)

        if len(combined_features) < k // 2:
            combined_features = list(mi_top_k | rf_top_k)[:k]

        print(f"\n--- Top-K: {k}, Combined Features: {combined_features} ---")

        for size in range(2, len(combined_features) + 1):
            for subset in combinations(combined_features, size):
                subset = list(subset)
                X_subset = X_scaled_df[subset]

                X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.3, random_state=RANDOM_STATE)

                iforest = IsolationForest(contamination=y_train.mean(), random_state=RANDOM_STATE)
                iforest.fit(X_train)
                y_pred = iforest.predict(X_test)
                y_pred = np.where(y_pred == -1, 1, 0)

                auc = roc_auc_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)
                accuracy = accuracy_score(y_test, y_pred)

                row = {
                        "Top_K": k,
                        "Feature_Combination_Size": size,
                        "Features": subset,
                        "ROC_AUC": auc,
                        "Precision": precision,
                        "Recall": recall,
                        "F1_Score": f1,
                        "Accuracy": accuracy
                    }

                writer.writerow(row)

                print(f"[Top_K={k} | {size}-features] Features={subset} | AUC={auc:.4f}, F1={f1:.4f}, Acc={accuracy:.4f}")


print(f"\nSaved results to {OUTPUT_FILE}")